## Structured Output

Models can be requested to provide their response in a format matching a given schema. This is useful ensuring the output can be easily parsed and used in subsequent processing. Langchain supports multiple schema types and methods for enforcing structured output

### Pydantic 

Pydantic models provide the richest feature set with field validation, descriptions, and nested structures.

In [8]:
import os 
from langchain.chat_models import init_chat_model

os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")

model=init_chat_model("groq:qwen/qwen3-32b")

In [ ]:
response=model.invoke("Hello how many requests can i put on this LLM as a free user")
print(response.content)

## concept of field validation

pydantic on its own does field validation for all the fields like title should have string year should be string and etc it evaluates individual fields on its own account

In [ ]:
from pydantic import BaseModel,Field
class Movie(BaseModel):
    title:str=Field(description="The title of the movie") 
    year:str=Field(description="This year movie was released")
    director:str=Field(description="Director of the movie")
    ratings:float=Field(description="Movie's ratings out of ten")


In [11]:
model_structure=model.with_structured_output(Movie)
model_structure


_ChatModelBinding(bound=ChatGroq(metadata={'lc_versions': {'langchain-core': '1.4.7', 'langchain': '1.3.9'}}, output_version=None, profile={'name': 'Qwen3 32B', 'release_date': '2024-12-23', 'last_updated': '2024-12-23', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 40960, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x00000263E57D5810>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x00000263E57D6710>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': {'properties': {'title': 

In [ ]:
response=model_structure.invoke("Provide details of the movie Oppenheimer")
response

#What we basically did here was structure the output to our needs and cutting out the rest of the useless info



Movie(title='Oppenheimer', year='2023', director='Christopher Nolan', ratings=8.8)

In [16]:
response=model.with_structured_output(Movie, include_raw=True)
response


{
  raw: _ChatModelBinding(bound=ChatGroq(metadata={'lc_versions': {'langchain-core': '1.4.7', 'langchain': '1.3.9'}}, output_version=None, profile={'name': 'Qwen3 32B', 'release_date': '2024-12-23', 'last_updated': '2024-12-23', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 40960, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x00000263E57D5810>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x00000263E57D6710>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': {'properties': {

## NESTED

In [17]:
from pydantic import BaseModel,Field

class Actor(BaseModel):
    name:str
    role:str

class Movie(BaseModel):
    title:str
    year:int
    cast:list[Actor]
    genres:list[str]
    budget:float
model_ss=model.with_structured_output(Movie)

response=model_ss.invoke("Details about the movie oppenheimer")
response

Movie(title='Oppenheimer', year=2023, cast=[Actor(name='Cillian Murphy', role='J. Robert Oppenheimer'), Actor(name='Emily Blunt', role='Kitty Oppenheimer'), Actor(name='Matt Damon', role='Lewis Strauss')], genres=['Biography', 'Drama', 'History'], budget=100000000.0)

# TypeDict

It provides a simpler alternatives using Python's built-in typing, ideal when you dont need runtime validation that pydantic provides

In [26]:
from typing_extensions import TypedDict,Annotated

class movie(TypedDict):
    title:Annotated[str,...,"The title of the movie"]
    year:Annotated[int,...,"The year movie was released"]
    director:Annotated[str,...,"Director of the movie"]
    rating:Annotated[float,...,"The movie's ratings out of 10"]

model_typeddict=model.with_structured_output(movie)

response=model_typeddict.invoke("Details of the movie Avengers")
response
    

{'director': 'Joss Whedon', 'rating': 8, 'title': 'Avengers', 'year': 2012}

We can check the profile of our base model that is the qwen model with no output editing

But we cant do the same for our pydantic or typeddict edited model

In [ ]:
model.profile

{'name': 'Qwen3 32B',
 'release_date': '2024-12-23',
 'last_updated': '2024-12-23',
 'open_weights': True,
 'max_input_tokens': 131072,
 'max_output_tokens': 40960,
 'text_inputs': True,
 'image_inputs': False,
 'audio_inputs': False,
 'video_inputs': False,
 'text_outputs': True,
 'image_outputs': False,
 'audio_outputs': False,
 'video_outputs': False,
 'reasoning_output': True,
 'tool_calling': True,
 'attachment': False,
 'temperature': True}